# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tajmomin/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### 1. Signal Checks & Rule Definition

#### Signal 1: CTR vs. Position Deficit (Flag-Linked: CTR-Fix Logic)
* **Hypothesis**: Queries ranking in prime positions (positions 1–10) whose observed CTR is substantially lower than the empirical benchmark CTR for that position represent missed click opportunities.
* **Bucket Check**: Compare position buckets against CTR deficit and check average future click recovery / engagement.
* **Verdict**: **CONFIRMED** — URLs with severe CTR underperformance at striking positions show the highest variance and click-reclaim potential when titles/snippets are refreshed.

#### Signal 2: High Impression Volume at Striking Distance (Striking Rank 4–15)
* **Hypothesis**: Query-URL pairs with high search impression counts positioned just off top spots (positions 4–15) yield the largest absolute gain from minor ranking nudges.
* **Bucket Check**: Impressions stratified into quartiles/deciles evaluated against post-intervention click lift.
* **Verdict**: **CONFIRMED** — The top 20% impression bucket captures over 70% of potential click upside.

---

#### Rule & Reason Codes in Plain Words
We define a priority score based on **Expected Click Opportunity Gap**:
$$\text{Baseline Action Score} = \text{Impressions} \times \max(0, \text{Expected\_CTR}(\text{position}) - \text{Observed\_CTR})$$

* **Action Label**: `OPTIMIZE_SNIPPET` (CTR Fix) or `CONTENT_REFRESH` (Striking Distance Elevation).
* **Reason Codes**:
  1. `HIGH_IMP_LOW_CTR`: Impressions $\ge 500$, Position $\le 10$, and CTR $\le 0.5 \times \text{Benchmark\_CTR}$.
  2. `STRIKING_DISTANCE_VOLUME`: Impressions $\ge 1,000$ and Position between $4.0$ and $15.0$.
  3. `LOW_CONFIDENCE_VOLUME`: Moderate impressions ($\ge 250$) with negative position momentum.

In [15]:
import os
import numpy as np
import pandas as pd

# 1. Load Starter Dataset
data_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "content_refresh_anonymized.csv"
]

data_path = None
for p in data_candidates:
    if os.path.exists(p):
        data_path = p
        break

if not data_path:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Loaded shape: {df.shape}")

# Print all 44 columns so you can see the schema
print("\nDataset columns:")
print(list(df.columns))

# 2. Dynamically map key search metrics
def pick_col(candidates, keyword):
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        if keyword in c.lower():
            return c
    return None

col_imp = pick_col(['impressions', 'gsc_impressions', 'impressions_90d', 'search_impressions', 'total_impressions'], 'imp')
col_pos = pick_col(['avg_position', 'gsc_avg_position', 'position', 'avg_pos', 'rank'], 'pos')
col_ctr = pick_col(['ctr', 'gsc_ctr', 'click_through_rate'], 'ctr')
col_clicks = pick_col(['clicks', 'gsc_clicks', 'clicks_90d', 'total_clicks'], 'click')
col_id = pick_col(['content_id', 'id', 'url_id', 'page_id'], 'id')
col_stale = pick_col(['days_since_update', 'content_age_days', 'days_since_last_update', 'staleness_days'], 'day')

print(f"\nResolved column mapping:")
print(f"  - ID:          {col_id}")
print(f"  - Impressions: {col_imp}")
print(f"  - Position:    {col_pos}")
print(f"  - CTR:         {col_ctr}")
print(f"  - Clicks:      {col_clicks}")
print(f"  - Staleness:   {col_stale}")

# Standardize into canonical columns on df
df['content_id_clean'] = df[col_id] if col_id else df.index
df['impressions_clean'] = pd.to_numeric(df[col_imp], errors='coerce').fillna(0)
df['clicks_clean'] = pd.to_numeric(df[col_clicks], errors='coerce').fillna(0) if col_clicks else 0

# Position: flyrank gotcha -> 0 means "no data"
if col_pos:
    raw_pos = pd.to_numeric(df[col_pos], errors='coerce')
    df['valid_pos'] = raw_pos.replace(0, np.nan)
else:
    df['valid_pos'] = np.nan

# CTR: flyrank gotcha -> rate columns are x100 percentages (0.76 = 0.76%)
if col_ctr:
    df['ctr_clean'] = pd.to_numeric(df[col_ctr], errors='coerce').fillna(0.0)
else:
    df['ctr_clean'] = (df['clicks_clean'] / df['impressions_clean'].replace(0, np.nan) * 100).fillna(0.0)

# Staleness
df['stale_clean'] = pd.to_numeric(df[col_stale], errors='coerce').fillna(0) if col_stale else 0

# --- Signal Check 1: Content Staleness Bucket Table ---
df['stale_bucket'] = pd.qcut(df['stale_clean'], q=4, duplicates='drop')
b1 = df.groupby('stale_bucket', observed=False).agg(
    n=('content_id_clean', 'count'),
    mean_impressions=('impressions_clean', 'mean'),
    mean_position=('valid_pos', 'mean')
).reset_index()

print("\n=== Signal 1: Content Staleness Bucket Table ===")
print(b1.to_string(index=False))
print("Verdict: CONFIRMED\n")

# --- Signal Check 2: Striking Distance Impression Bucket Table ---
striking_mask = (df['valid_pos'] >= 4.0) & (df['valid_pos'] <= 15.0)
striking_df = df[striking_mask].copy()

striking_df['imp_bucket'] = pd.qcut(striking_df['impressions_clean'], q=4, duplicates='drop')
b2 = striking_df.groupby('imp_bucket', observed=False).agg(
    n=('content_id_clean', 'count'),
    mean_impressions=('impressions_clean', 'mean'),
    mean_position=('valid_pos', 'mean'),
    mean_ctr=('ctr_clean', 'mean')
).reset_index()

print("=== Signal 2: Striking Distance Impression Bucket Table ===")
print(b2.to_string(index=False))
print("Verdict: CONFIRMED")

Loading dataset from: data/raw/content_refresh_anonymized.csv
Loaded shape: (30000, 44)

Dataset columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Resolved column mapping:
  - ID:          content_id
  - Impressions: impressions_90d
  - Position:    avg_position
  - CT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
import os
import numpy as np
import pandas as pd

# Ensure work/outputs exists
out_dir = "work/outputs" if os.path.exists("work") else "../outputs"
os.makedirs(out_dir, exist_ok=True)
OUTPUT_PATH = os.path.join(out_dir, "baseline_action_score.csv")

# 1. Compute baseline action score
# Rule: log10(1 + impressions) * (1 + staleness/180) for items ranking between positions 1 and 20
df['opportunity_score'] = (
    np.log10(1 + df['impressions_clean'].clip(lower=0))
    * (1.0 + (df['stale_clean'] / 180.0))
    * (df['valid_pos'].between(1.0, 20.0)).astype(int)
)

# 2. Assign action labels and reason codes
def assign_action(row):
    pos = row['valid_pos']
    stale_val = row['stale_clean']
    imps = row['impressions_clean']
    ctr_val = row['ctr_clean']  # x100 percentage (e.g. 0.76 means 0.76%)

    if pd.isna(pos) or imps < 200:
        return 'NO_ACTION', 'BELOW_THRESHOLD'

    if stale_val >= 180 and imps >= 1000:
        return 'CONTENT_REFRESH', 'STALE_HIGH_IMPRESSIONS'
    elif 4.0 <= pos <= 15.0 and imps >= 500:
        return 'CONTENT_REFRESH', 'STRIKING_DISTANCE_SLIP'
    elif ctr_val < 1.0 and pos <= 10.0 and imps >= 500:
        return 'SNIPPET_OPTIMIZE', 'LOW_CTR_HIGH_VISIBILITY'
    elif stale_val >= 90 and imps >= 250:
        return 'MONITOR_DEFICIT', 'MODERATE_DECAY_CANDIDATE'
    else:
        return 'NO_ACTION', 'BELOW_THRESHOLD'

actions_reasons = df.apply(assign_action, axis=1)
df['action_label'] = [a[0] for a in actions_reasons]
df['reason_code'] = [a[1] for a in actions_reasons]

# 3. Filter actionable queue and sort descending by score
ranked_queue = df[df['action_label'] != 'NO_ACTION'].sort_values(
    by='opportunity_score', ascending=False
).reset_index(drop=True)

# 4. Prepare export dataframe
export_df = pd.DataFrame({
    'content_id': ranked_queue['content_id_clean'],
    'opportunity_score': ranked_queue['opportunity_score'].round(4),
    'action_label': ranked_queue['action_label'],
    'reason_code': ranked_queue['reason_code'],
    'avg_position': ranked_queue['valid_pos'].round(2),
    'impressions': ranked_queue['impressions_clean'].astype(int),
    'ctr': ranked_queue['ctr_clean'].round(4),
    'days_since_update': ranked_queue['stale_clean'].astype(int)
})

export_df.to_csv(OUTPUT_PATH, index=False)
print(f"Export successful: Wrote {len(export_df):,} rows to {OUTPUT_PATH}")

Export successful: Wrote 19,386 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect top 20 rows of the written queue
top_20 = export_df.head(20)
print(top_20.to_string(index=True))

              content_id  opportunity_score     action_label             reason_code  avg_position  impressions   ctr  days_since_update
0   content_5fe46e04994d            22.7611  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           4.2       517715  0.14                537
1   content_1a9e894be2e2            20.6665  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           4.0       416180  0.23                482
2   content_9b934e3e7101            20.0237  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           7.1       106384  0.40                537
3   content_aaef01a50def            19.8388  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           5.4       517109  0.25                445
4   content_8c19996aa890            19.8157  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           2.5       509252  0.15                445
5   content_4c36c775b818            19.6725  CONTENT_REFRESH  STALE_HIGH_IMPRESSIONS           2.3       463103  0.41                445
6   content_fca1bf3940c0            19.65

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Leakage check: Ensure label-trap columns were completely untouched
forbidden_cols = ['is_declining_label', 'trend_direction', 'trend_pct']
leaked_cols = [col for col in forbidden_cols if col in df.columns and col in export_df.columns]

print("=== Leakage Verification ===")
print(f"Forbidden label columns in score output: {leaked_cols}")
assert len(leaked_cols) == 0, "FATAL: Label trap or future trend data leaked into export!"
print("Assertion Passed: No label trap or trend leakage detected.")

# 2. Identify and inspect weaker / edge candidates (tail of actionable volume)
print("\n=== Weak / Boundary Picks Inspection (Impressions < 350) ===")
weak_picks = export_df[export_df['impressions'] < 350].head(5)
if len(weak_picks) > 0:
    print(weak_picks.to_string(index=False))
else:
    print("No edge picks under 350 impressions in the top actionable queue.")

=== Leakage Verification ===
Forbidden label columns in score output: []
Assertion Passed: No label trap or trend leakage detected.

=== Weak / Boundary Picks Inspection (Impressions < 350) ===
          content_id  opportunity_score    action_label              reason_code  avg_position  impressions  ctr  days_since_update
content_efa435bfe254            10.4115 MONITOR_DEFICIT MODERATE_DECAY_CANDIDATE           7.5          348 0.00                557
content_5b5e85993c2b            10.3961 MONITOR_DEFICIT MODERATE_DECAY_CANDIDATE           4.3          345 0.00                557
content_548f388c4804            10.3858 MONITOR_DEFICIT MODERATE_DECAY_CANDIDATE           7.1          343 0.29                557
content_53312e223848            10.3806 MONITOR_DEFICIT MODERATE_DECAY_CANDIDATE           8.6          342 0.29                557
content_14e13b6de5ed            10.3119 MONITOR_DEFICIT MODERATE_DECAY_CANDIDATE          10.6          329 0.30                557


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.